# MatPES MLIP Analysis

Analyses force-error statistics across all potentials using the merged MatPES PBE results.

| Section | Purpose |
|---|---|
| 0 — Imports & Load Data | Load `all_results` and `all_dfs` |
| 1 — Configuration | Thresholds, model exclusions |
| 2 — Fraction Tables | Build fraction (%) tables for each angle cut |
| 3 — Text Summary Table | "p1 / p20" formatted table for inspection |
| 4 — Split Triangle Heatmap | Per-cell: lower = Δθ < 1°, upper = Δθ < 20° |
| 5 — ΔF-Only Fraction Heatmap | Fraction with \|ΔF\| < threshold (no angle filter) |
| 6 — Density & CDF Config | MLIP selection, filtered dict, CDF computation |
| 7 — \|ΔF\| vs Δθ Density | 2-D density: force error vs angular error |
| 8 — \|ΔF\| vs F_DFT Density | 2-D density: force error vs DFT force magnitude |
| 9 — Δθ vs F_DFT Density | 2-D density: angular error vs DFT force magnitude |
| 10 — CDF of \|ΔF\| | Cumulative distribution of force magnitude error |
| 11 — CDF of \|Δθ\| | Cumulative distribution of angular error |
| 12 — Large-Error Fraction Heatmap | % atoms with \|ΔF\| **>** threshold (log-scale cbar) |
| 13 — Two-Regime Error Panels | Near-eq / far-from-eq absolute + relative error fractions |
| 14 — F_DFT Distribution (single threshold) | F_DFT profile of atoms with large \|ΔF\| |
| 15 — F_DFT Distribution (loop) | Same as 14 for multiple \|ΔF\| thresholds |
| 16 — F_DFT Fraction Table | % atoms with \|F_DFT\| > threshold (force distribution) |
| 17 — F_DFT-Conditioned MAE/RMSE | MAE/RMSE of \|ΔF\| for atoms with \|F_DFT\| > threshold |
| 18 — Δθ Fraction Table | % atoms with \|Δθ\| < threshold (imshow heatmap) |
| 19 — Δθ Conditioned MAE/RMSE | MAE/RMSE of \|Δθ\| for atoms with \|Δθ\| < threshold |
| 20 — Error Histograms | \|ΔF\| (log-log) and \|Δθ\| (linear-log) histograms |
| 21 — Theta Regime Panels | Near-eq / far-from-eq Δθ fraction panels |
| 22 — bad attom indices | returns indices of the original data set depending on what region the user ones |

**Input files:**
- `all_results_PBE_all_data.json` — `{model: {"F_dft": [...], "deltaF": [...], "deltaTheta": [...]}}`
- `all_dfs.json` — merged per-structure DataFrames

---
## Section 0 — Imports & Load Data

In [ ]:
import json, os
import numpy as np
import pandas as pd

import sys
sys.path.insert(0, '../../scripts')

from matpes_frac_analysis import (
    frac_percent,
    build_frac_table,
    build_dF_frac_table,
    split_triangle_heatmap,
    single_heatmap,
)

BASE = "./results"  # relative to examples/lyc/

all_results = {}
for file in os.listdir(BASE):
    if file.startswith("all_results_") and file.endswith("_PBE.json"):
        model_name = file[len("all_results_"):-len("_PBE.json")]
        with open(os.path.join(BASE, file)) as f:
            all_results[model_name] = json.load(f)

print("Models in all_results:", list(all_results.keys()))

---
## Section 1 — Configuration

- `DF_CUTS` — force-error thresholds to evaluate (eV/Å)
- `ANGLE_CUTS` — angular thresholds (degrees); `[1, 20]` gives the two standard cuts
- `FDFT_MIN` — minimum |F_DFT| for an atom to be included (filters near-zero DFT forces)
- `EXCLUDE_MODELS` — models to drop from all tables and plots (e.g. r2SCAN potentials)

In [ ]:
DF_CUTS = [0.01, 0.02, 0.05, 0.07, 0.1, 0.2, 0.5]   # eV/Å
ANGLE_CUTS = [1, 20]                                   # degrees
FDFT_MIN   = 0.01                                      # eV/Å

# Models to exclude from all tables and plots (e.g. r2SCAN potentials)
EXCLUDE_MODELS = [
    "TensorNET_matpes_R2SCAN",
    "m3gnet_matpes_r2scan",
]

# Column labels shared across all heatmaps
COL_LABELS = [
    r"|ΔF| < 0.01 eV/Å",
    r"|ΔF| < 0.02 eV/Å",
    r"|ΔF| < 0.05 eV/Å",
    r"|ΔF| < 0.07 eV/Å",
    r"|ΔF| < 0.1 eV/Å",
    r"|ΔF| < 0.2 eV/Å",
    r"|ΔF| < 0.5 eV/Å",
]

# Filtered model list (preserves the order from all_results)
models = [m for m in all_results if m not in EXCLUDE_MODELS]
print("Models included:", models)

---
## Section 2 — Fraction Tables

`build_frac_table` returns one DataFrame per angle cut:
- `tables[1]`  — fraction (%) of atoms with |ΔF| < threshold **and** Δθ < 1°
- `tables[20]` — fraction (%) of atoms with |ΔF| < threshold **and** Δθ < 20°

`build_dF_frac_table` gives the ΔF-only fraction (no angle filter).

In [ ]:
all_results["mace"].keys()

In [ ]:
# Fraction tables for each angle cut (rows = models, cols = DF_CUTS)
tables = build_frac_table(all_results, DF_CUTS, ANGLE_CUTS, fdf_min=FDFT_MIN)

# Restrict to included models
for ac in ANGLE_CUTS:
    tables[ac] = tables[ac].loc[models]

# ΔF-only fraction table (no angle filter)
df_frac = build_dF_frac_table(all_results, DF_CUTS, fdf_min=FDFT_MIN).loc[models]

# Arrays for the split-triangle heatmap
# lower triangle ← fractions at Δθ < 1°
# upper triangle ← fractions at Δθ < 20°
vals_lower = tables[1].values    # shape (n_models, n_dF_cuts)
vals_upper = tables[20].values

print(f"Table shape: {vals_lower.shape}  ({len(models)} models × {len(DF_CUTS)} thresholds)")
tables[1]

---
## Section 3 — Text Summary Table

Each cell shows `"p1 / p20"` where:
- `p1`  = fraction (%) at |ΔF| < threshold **and** Δθ < 1°
- `p20` = fraction (%) at |ΔF| < threshold **and** Δθ < 20°

In [ ]:
df_summary = pd.DataFrame(index=models)

for thr in DF_CUTS:
    col_name = f"|ΔF|<{thr} (Δθ<1°/20°) [%]"
    df_summary[col_name] = [
        f"{tables[1].loc[m, thr]:.2f} / {tables[20].loc[m, thr]:.2f}"
        for m in models
    ]

df_summary

---
## Section 4 — Split Triangle Heatmap

Each cell is split diagonally:
- **Lower triangle** — fraction (%) with |ΔF| < threshold **and** Δθ < 1°
- **Upper triangle** — fraction (%) with |ΔF| < threshold **and** Δθ < 20°

Higher values = better agreement with DFT.

In [ ]:
title = (
    "% atoms with |ΔF| < threshold\n"
    "and Δθ < 1° \\ 20°\n"
    "(higher = better) ↑\n"
    "(F_DFT > 0.01 eV/Å)"
)

fig, ax = split_triangle_heatmap(
    vals_lower, vals_upper,
    row_labels       = models,
    col_labels       = COL_LABELS,
    title            = title,
    show_row_labels= True,
    cmap_lower       = "viridis",
    cmap_upper       = "viridis",
    annotate         = True,
    fmt              = "{:.1f}",
    textsize         = 6,
    addsize          = 1,
    cbar_label_lower = "[%] (<1°)",
    cbar_label_upper = "[%] (<20°)",
    figsize          = (3.5, 5.5),
    gap              = 0.02,
    gap_between_cbars= 0.15,
    right=0.80, bottom=0.0225, top=1.0,
    savepath         = "./jsonfiles_for_summary_table/deltafdeltaangletable.svg",
)

---
## Section 5 — ΔF-Only Fraction Heatmap

Fraction (%) of atoms with |ΔF| < threshold — **no angle filter applied**.
Atoms with |F_DFT| ≤ 0.01 eV/Å are excluded.

In [ ]:
fig, ax = single_heatmap(
    data       = df_frac.values,
    row_labels = df_frac.index.tolist(),
    col_labels = COL_LABELS,
    show_row_labels = True, 
    title      = (
        "% atoms with |ΔF| < threshold\n"
        "(higher = better) ↑\n"
        "(F_DFT > 0.01 eV/Å)"
    ),
    cmap       = "viridis",
    annotate   = True,
    fmt        = "{:.1f}",
    textsize   = 8,
    addsize    = 0,
    cbar       = True,
    cbar_label = "[%]",
    figsize    = (3.5, 5.5),
    gap        = 0.025,
    right      = 0.80,
    bottom     = 0.02,
    top        = 1.0,
    savepath   = "./jsonfiles_for_summary_table/df_frac_heatmap.svg",
)

---
## Section 6 — Density & CDF Configuration

`MLIP_SHOW` selects which single potential is shown in the density plots (Sections 7–9).  
All models are shown together in the CDF plots (Sections 10–11).

The style dicts (`PAPER_STYLE_DENSITY`, `PAPER_STYLE_CDF`) contain all tested font/tick sizes.

In [ ]:
import matplotlib.pyplot as plt

from mlip_cdf_density_plots import (
    build_cdf_from_all_results,
    panel_abs_dF_vs_dtheta_cond_on_Fdft,
    panel_Fdft_vs_abs_dF,
    panel_Fdft_vs_dtheta,
    plot_cdf_with_inset_on_ax,
    PAPER_STYLE_DENSITY,
    PAPER_STYLE_CDF,
)

# Which single MLIP to show in density plots (Sections 7–9)
MLIP_SHOW = "mace"   # change to any key in all_results

# Dict restricted to included models (no r2SCAN)
all_results_filtered = {m: all_results[m] for m in models}

# Pre-compute CDFs for all included models — used in Sections 10 & 11
cdf_results = build_cdf_from_all_results(
    all_results_filtered,
    f_dft_thr=FDFT_MIN,
    use_abs_f_dft=True,
)

print(f"Density plots will show : {MLIP_SHOW}")
print(f"CDF computed for        : {list(cdf_results.keys())}")

---
## Section 7 — |ΔF| vs Δθ Density

2-D density of force-magnitude error vs angular error, conditioned on |F_DFT| > FDFT_MIN.  
Shows how correlated the two error types are for the selected MLIP.

In [ ]:
fig = plt.figure(figsize=(5, 4.5), dpi=350)
gs  = fig.add_gridspec(1, 1)

panel_abs_dF_vs_dtheta_cond_on_Fdft(
    fig, gs[0, 0],
    all_results_filtered, MLIP_SHOW,
    fdft_threshold = FDFT_MIN,
    xlim           = (None, 1e3),
    inner_hspace   = 0.2,
    **PAPER_STYLE_DENSITY,
)

plt.savefig(f"dF_vs_dtheta_{MLIP_SHOW}.svg", bbox_inches="tight", dpi=350)
plt.show()

---
## Section 8 — |ΔF| vs F_DFT Density

Force magnitude error vs DFT force magnitude. Shows whether errors are
larger at high-force atoms (large forces are harder to predict).

In [ ]:
fig = plt.figure(figsize=(5, 4.5), dpi=350)
gs  = fig.add_gridspec(1, 1)

panel_Fdft_vs_abs_dF(
    fig, gs[0, 0],
    all_results_filtered, MLIP_SHOW,
    force_threshold = FDFT_MIN,
    xlim            = (None, 1000),
    **PAPER_STYLE_DENSITY,
)

plt.savefig(f"Fdft_vs_dF_{MLIP_SHOW}.svg", bbox_inches="tight", dpi=350)
plt.show()

---
## Section 9 — Δθ vs F_DFT Density

Angular error vs DFT force magnitude. Near-zero DFT forces are excluded
(angular error is undefined / ill-conditioned there).

In [ ]:
fig = plt.figure(figsize=(5, 4.5), dpi=350)
gs  = fig.add_gridspec(1, 1)

panel_Fdft_vs_dtheta(
    fig, gs[0, 0],
    all_results_filtered, MLIP_SHOW,
    force_threshold = FDFT_MIN,
    xlim            = (None, 1000),
    **PAPER_STYLE_DENSITY,
)

plt.savefig(f"Fdft_vs_dtheta_{MLIP_SHOW}.svg", bbox_inches="tight", dpi=350)
plt.show()

---
## Section 10 — CDF of |ΔF|

Cumulative distribution of force magnitude error across **all included models**.  
The inset zooms into the high-CDF tail (> 0.97) to separate models that are otherwise overlapping.

In [ ]:
fig = plt.figure(figsize=(4.5, 3.5), dpi=350)
gs  = fig.add_gridspec(1, 1)

plot_cdf_with_inset_on_ax(
    fig, gs[0, 0],
    cdf_results,
    kind         = "dF",
    title        = (
        r"CDF of $|\Delta F|$ (all models)" "\n"
        rf"$|F_{{\mathrm{{DFT}}}}| > {FDFT_MIN}$ eV/Å only"
    ),
    xlabel       = r"$|\Delta F|$ (eV/$\mathrm{\AA}$)",
    xlim_main    = (-0.02, 2.5),
    xlim_inset   = (0.5, 4.0),
    ylim_inset   = (0.97, 1.0),
    inset_bbox   = (0.25, 0.04, 1, 1),
    legend_bbox  = (1.02, 0.5),
    legend_loc   = "center left",
    legend_ncols = 1,
    show_legend  = True,
    **PAPER_STYLE_CDF,
)

plt.savefig("cdf_dF.svg", bbox_inches="tight", dpi=350)
plt.show()

---
## Section 11 — CDF of |Δθ|

Cumulative distribution of angular error across **all included models**.  
The inset zooms into the mid-CDF range (35°–55°) where model separation is clearest.

In [ ]:
fig = plt.figure(figsize=(4.5, 3.5), dpi=350)
gs  = fig.add_gridspec(1, 1)

plot_cdf_with_inset_on_ax(
    fig, gs[0, 0],
    cdf_results,
    kind         = "theta",
    title        = (
        r"CDF of $|\Delta \theta|$ (all models)" "\n"
        rf"$|F_{{\mathrm{{DFT}}}}| > {FDFT_MIN}$ eV/Å only"
    ),
    xlabel       = r"$|\Delta \theta|$ (deg)",
    xlim_main    = (-2.0, 182),
    xlim_inset   = (35, 55),
    ylim_inset   = (0.65, 0.85),
    inset_bbox   = (0.25, 0.06, 1, 1),
    legend_bbox  = (1.02, 0.5),
    legend_loc   = "center left",
    legend_ncols = 1,
    show_legend  = True,
    **PAPER_STYLE_CDF,
)

plt.savefig("cdf_dtheta.svg", bbox_inches="tight", dpi=350)
plt.show()

---
## Section 12 — Large-Error Fraction Heatmap

Fraction (%) of atoms where **|ΔF| > threshold** (opposite of Section 5).  
A log-scale colorbar is used because fractions span several orders of magnitude.  
Lower values = better (fewer large errors).

In [ ]:
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from heatmap_table import (
    create_figure, setup_frame, setup_ticks_and_labels,
    draw_rectangular_column, add_colorbar,
)
from matpes_frac_analysis import build_dF_frac_larger_table

# ── Config ─────────────────────────────────────────────────────────────────
DF_FRAC_THRESH_LARGE = [0.5, 1, 2, 3, 4, 5, 7, 10]   # eV/Å

text_size = 11
figsize   = (6.0, 4.5)

# ── Compute ─────────────────────────────────────────────────────────────────
df_frac_larger = build_dF_frac_larger_table(
    all_results_filtered, DF_FRAC_THRESH_LARGE, fdf_min=FDFT_MIN
)

display(df_frac_larger)

# ── Plot ─────────────────────────────────────────────────────────────────────
col_labels_large = [rf"$|\Delta F| > {thr:g}$ eV/Å" for thr in DF_FRAC_THRESH_LARGE]
models_large     = df_frac_larger.index.tolist()
nrows_l, ncols_l = df_frac_larger.shape

vals = df_frac_larger.values.astype(float)
norm = LogNorm(
    vmin=np.nanmin(vals[vals > 0]) if np.any(vals > 0) else 1e-2,
    vmax=np.nanmax(vals),
)
cmap = plt.cm.viridis_r

fig, ax, (cax,) = create_figure(
    ncols_l, n_colorbars=1,
    figsize=figsize, dpi=350,
    wspace=0.35, colorbar_nudge=0.10,
)
ax.set_xlim(0, ncols_l)
ax.set_ylim(0, nrows_l)
ax.set_aspect("equal")

for j, col in enumerate(df_frac_larger.columns):
    draw_rectangular_column(
        ax, col_idx=j, nrows=nrows_l,
        vals=df_frac_larger[col].values,
        cmap=cmap, norm=norm,
        fmt="{:.2f}", text_size=text_size,
    )

setup_frame(ax, ncols_l, nrows_l)
setup_ticks_and_labels(
    ax,
    ncols=ncols_l, nrows_data=nrows_l, nrows_total=nrows_l,
    row_labels=models_large, col_labels=col_labels_large,
    title=(
        r"% atoms with $|\Delta F|$ > threshold (eV/Å)" + "\n"
        r"$F_{\mathrm{DFT}}$ > 0.01 eV/Å  (lower = better ↓)"
    ),
    xlabel="", extra_row_label=None, text_size=text_size,
)

cb = add_colorbar(fig, ax, cax, cmap=cmap, norm=norm,
                  label="[%]", text_size=text_size)
cb.set_label("[%]", fontsize=text_size + 1, rotation=270, labelpad=12)
cb.ax.yaxis.set_major_locator(ticker.LogLocator(base=10, numticks=6))
cb.ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())
cb.ax.tick_params(which="minor", length=2)

plt.tight_layout()
fig.savefig("fraction_table_large_errors.svg", bbox_inches="tight", dpi=350, pad_inches=0.05)
plt.show()

---
## Section 13 — Two-Regime Error Panels

Atoms are split by |F_DFT| into two regimes:
- **Panel A** — near-equilibrium: |F_DFT| < 1 eV/Å
- **Panel B** — far-from-equilibrium: |F_DFT| ≥ 1 eV/Å

For each regime, four heatmaps are produced:
1. Panel A — absolute thresholds (`|ΔF| < x`)
2. Panel A — relative thresholds (`r = |ΔF|/(|F_DFT|+ε) < x`)
3. Panel B — relative thresholds
4. Panel B — absolute thresholds

In [ ]:
from heatmap_table import plot_fraction_panel
from matpes_frac_analysis import build_regime_panels

# ── Config ──────────────────────────────────────────────────────────────────
ABS_THRESH = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
REL_THRESH = [0.01, 0.05, 0.10, 0.20, 0.3, 0.4, 0.50, 1, 2]
THRESHOLD  = 1.0    # eV/Å boundary between panels
EPS        = 0.01   # stabiliser for relative error

fig_w        = 6.5
fig_h        = 5.98
text_size    = 13
cbar_gap     = 0.115
cbar_lbl_pad = 15

# ── Build DataFrames ─────────────────────────────────────────────────────────
df_panel_A, df_panel_B = build_regime_panels(
    all_results_filtered,
    abs_thresh=ABS_THRESH,
    rel_thresh=REL_THRESH,
    threshold=THRESHOLD,
    eps=EPS,
)

print(f"Panel A — |F_DFT| < {THRESHOLD} eV/Å")
display(df_panel_A)
print(f"Panel B — |F_DFT| ≥ {THRESHOLD} eV/Å")
display(df_panel_B)

# Column subsets to pass to plot_fraction_panel
abs_cols = (["N atoms", "Frac of all atoms [%]"]
            + [f"|ΔF| < {thr} [%]" for thr in ABS_THRESH])
rel_cols = (["N atoms", "Frac of all atoms [%]"]
            + [f"r < {thr} [%]"    for thr in REL_THRESH])

frac_avg_A = df_panel_A["Frac of all atoms [%]"].mean()

# ── 4 heatmaps (each their own figure) ───────────────────────────────────────

plot_fraction_panel(
    df_panel_A[abs_cols],
    fmt="{:.2f}",
    panel_title=f"Near-equilibrium (|F_DFT| < {THRESHOLD} eV/Å) — absolute\n({frac_avg_A:.1f}% of all atoms)",
    fig_w=fig_w, fig_h=fig_h,
    text_size=text_size,
    cbar_gap=cbar_gap, cbar_label_pad=cbar_lbl_pad,
    show_regime_row=False,
)

plot_fraction_panel(
    df_panel_A[rel_cols],
    fmt="{:.2f}",
    panel_title=f"Near-equilibrium (|F_DFT| < {THRESHOLD} eV/Å) — relative\n({frac_avg_A:.1f}% of all atoms)",
    fig_w=fig_w, fig_h=fig_h,
    text_size=text_size,
    cbar_gap=cbar_gap, cbar_label_pad=cbar_lbl_pad,
    show_regime_row=False,
)

plot_fraction_panel(
    df_panel_B[rel_cols],
    fmt="{:.2f}",
    panel_title=f"Far-from-equilibrium (|F_DFT| ≥ {THRESHOLD} eV/Å) — relative\n({100 - frac_avg_A:.1f}% of all atoms)",
    mae_cmap=plt.cm.Purples, rmse_cmap=plt.cm.Oranges,
    fig_w=fig_w, fig_h=fig_h,
    text_size=text_size,
    save_path="far_from_equilibrium_summary_rel.svg",
    cbar_gap=cbar_gap, cbar_label_pad=cbar_lbl_pad,
    show_regime_row=False,
)

plot_fraction_panel(
    df_panel_B[abs_cols],
    fmt="{:.2f}",
    panel_title=f"Far-from-equilibrium (|F_DFT| ≥ {THRESHOLD} eV/Å) — absolute\n({100 - frac_avg_A:.1f}% of all atoms)",
    mae_cmap=plt.cm.Purples, rmse_cmap=plt.cm.Oranges,
    fig_w=fig_w, fig_h=fig_h,
    text_size=text_size,
    cbar_gap=cbar_gap, cbar_label_pad=cbar_lbl_pad,
    show_regime_row=False,
)

---
## Section 14 — F_DFT Distribution for Large-Error Atoms (single threshold)

For atoms with |ΔF| > `DF_THRESH`:
- First column: what percentage of all atoms these represent
- Remaining columns: fraction with |F_DFT| below each threshold

If large errors cluster at low F_DFT, the MLIP struggles with near-equilibrium atoms;
if they cluster at high F_DFT, the MLIP struggles with high-force environments.

In [ ]:
from matplotlib.colors import Normalize
from heatmap_table import (
    create_figure, setup_frame, setup_ticks_and_labels,
    draw_rectangular_column,
)
from matpes_frac_analysis import build_large_dF_fdft_table

# ── Config ───────────────────────────────────────────────────────────────────
DF_THRESH   = 10.0
FDFT_THRESH = [1, 2, 3, 5, 7, 10, 100]   # eV/Å
text_size   = 8
figsize14   = (6.5, 3.5)
cbar_w      = 0.018
cbar_gap14  = 0.075

# ── Compute ──────────────────────────────────────────────────────────────────
df14 = build_large_dF_fdft_table(
    all_results_filtered, df_thresh=DF_THRESH,
    fdft_thresh=FDFT_THRESH, fdf_min=FDFT_MIN,
)
display(df14)

# ── Plot ─────────────────────────────────────────────────────────────────────
first_col  = df14.columns[0]
pct_cols14 = [c for c in df14.columns if "|F_DFT|" in c]
models14   = df14.index.tolist()
nrows14    = len(models14)
ncols14    = len(pct_cols14) + 1

col_labels14 = [f"% of all\natoms (|ΔF|>{DF_THRESH:g})"] + [c.replace(" [%]", "") for c in pct_cols14]

vals_pct14 = df14[pct_cols14].values.astype(float)
norm_pct14 = Normalize(vmin=np.nanpercentile(vals_pct14, 5), vmax=np.nanpercentile(vals_pct14, 95))
cmap_pct14 = plt.cm.RdYlGn

vals_n14   = df14[first_col].values.astype(float)
norm_n14   = Normalize(vmin=np.nanmin(vals_n14), vmax=np.nanmax(vals_n14))
cmap_n14   = plt.cm.Greys

fig, ax, _ = create_figure(ncols14, n_colorbars=0, figsize=figsize14, dpi=350)
ax.set_xlim(0, ncols14)
ax.set_ylim(0, nrows14)
ax.set_aspect("equal")

draw_rectangular_column(ax, col_idx=0, nrows=nrows14,
    vals=vals_n14, cmap=cmap_n14, norm=norm_n14,
    fmt="{:.2f}", text_size=text_size)

for j, col in enumerate(pct_cols14):
    draw_rectangular_column(ax, col_idx=j + 1, nrows=nrows14,
        vals=df14[col].values, cmap=cmap_pct14, norm=norm_pct14,
        fmt="{:.1f}", text_size=text_size)

ax.plot([1, 1], [0, nrows14], color="black", lw=1.5, ls="--")
setup_frame(ax, ncols14, nrows14)
setup_ticks_and_labels(ax,
    ncols=ncols14, nrows_data=nrows14, nrows_total=nrows14,
    row_labels=models14, col_labels=col_labels14,
    title=rf"$|F_{{DFT}}|$ distribution for atoms with $|\Delta F| > {DF_THRESH:g}$ eV/Å",
    xlabel="", extra_row_label=None, text_size=text_size)

fig.canvas.draw()
ax_pos = ax.get_position()
x0 = ax_pos.x0 + ax_pos.width + 0.025

for k, (cmap_k, norm_k, label_k) in enumerate([
    (cmap_n14,   norm_n14,   "% of all atoms"),
    (cmap_pct14, norm_pct14, "Fraction [%]"),
]):
    cax = fig.add_axes([x0 + k * (cbar_w + cbar_gap14), ax_pos.y0, cbar_w, ax_pos.height])
    sm  = plt.cm.ScalarMappable(norm=norm_k, cmap=cmap_k)
    cb  = fig.colorbar(sm, cax=cax)
    cb.ax.tick_params(labelsize=text_size)
    cb.set_label(label_k, fontsize=text_size + 1, rotation=270, labelpad=12)

fig.savefig("large_dF_fdft_table.svg", bbox_inches="tight", dpi=350, pad_inches=0.05)
plt.show()

---
## Section 15 — F_DFT Distribution (loop over multiple |ΔF| thresholds)

Same analysis as Section 14, repeated for each threshold in `DF_THRESHOLDS`.
Produces one heatmap per threshold value.

In [ ]:
DF_THRESHOLDS = [0.5, 1.0, 5.0, 10.0]    # eV/Å — one heatmap per value
FDFT_THRESH15 = [0.1, 0.5, 1, 5, 10, 100]
text_size15   = 10
figsize15     = (6.5, 3.5)
cbar_w15      = 0.018
cbar_gap15    = 0.085

for df_thresh in DF_THRESHOLDS:
    df15 = build_large_dF_fdft_table(
        all_results_filtered, df_thresh=df_thresh,
        fdft_thresh=FDFT_THRESH15, fdf_min=FDFT_MIN,
    )
    print(f"\nAtoms with |ΔF| > {df_thresh:g} eV/Å — |F_DFT| distribution")
    display(df15)

    first_col15  = df15.columns[0]
    pct_cols15   = [c for c in df15.columns if "|F_DFT|" in c]
    models15     = df15.index.tolist()
    nrows15      = len(models15)
    ncols15      = len(pct_cols15) + 1
    col_labels15 = [f"% of all\natoms (|ΔF|>{df_thresh:g})"] + [c.replace(" [%]", "") for c in pct_cols15]

    vals_pct15 = df15[pct_cols15].values.astype(float)
    norm_pct15 = Normalize(vmin=np.nanpercentile(vals_pct15, 5), vmax=np.nanpercentile(vals_pct15, 95))
    cmap_pct15 = plt.cm.RdYlGn

    vals_n15   = df15[first_col15].values.astype(float)
    norm_n15   = Normalize(vmin=np.nanmin(vals_n15), vmax=np.nanmax(vals_n15))
    cmap_n15   = plt.cm.Greys

    fig, ax, _ = create_figure(ncols15, n_colorbars=0, figsize=figsize15, dpi=350)
    ax.set_xlim(0, ncols15)
    ax.set_ylim(0, nrows15)
    ax.set_aspect("equal")

    draw_rectangular_column(ax, col_idx=0, nrows=nrows15,
        vals=vals_n15, cmap=cmap_n15, norm=norm_n15,
        fmt="{:.2f}", text_size=text_size15)

    for j, col in enumerate(pct_cols15):
        draw_rectangular_column(ax, col_idx=j + 1, nrows=nrows15,
            vals=df15[col].values, cmap=cmap_pct15, norm=norm_pct15,
            fmt="{:.1f}", text_size=text_size15)

    ax.plot([1, 1], [0, nrows15], color="black", lw=1.5, ls="--")
    setup_frame(ax, ncols15, nrows15)
    setup_ticks_and_labels(ax,
        ncols=ncols15, nrows_data=nrows15, nrows_total=nrows15,
        row_labels=models15, col_labels=col_labels15,
        title=rf"$|F_{{DFT}}|$ distribution for atoms with $|\Delta F| > {df_thresh:g}$ eV/Å",
        xlabel="", extra_row_label=None, text_size=text_size15)

    fig.canvas.draw()
    ax_pos = ax.get_position()
    x0 = ax_pos.x0 + ax_pos.width + 0.025

    for k, (cmap_k, norm_k, label_k) in enumerate([
        (cmap_n15,   norm_n15,   f"% of all atoms (|ΔF|>{df_thresh:g})"),
        (cmap_pct15, norm_pct15, "Fraction [%]"),
    ]):
        cax = fig.add_axes([x0 + k * (cbar_w15 + cbar_gap15), ax_pos.y0, cbar_w15, ax_pos.height])
        sm  = plt.cm.ScalarMappable(norm=norm_k, cmap=cmap_k)
        cb  = fig.colorbar(sm, cax=cax)
        cb.ax.tick_params(labelsize=text_size15)
        cb.set_label(label_k, fontsize=text_size15 + 1, rotation=270, labelpad=12)

    plt.show()

---
## Section 16 — F_DFT Fraction Table

Fraction (%) of atoms with |F_DFT| **above** each threshold — shows the
DFT force-magnitude distribution across all atoms.  
No model-specific filtering: every model sees the same DFT forces.

In [ ]:
from matpes_frac_analysis import build_F_dft_frac_table, single_heatmap

# ── Config ────────────────────────────────────────────────────────────────────
FDFT_FRACS_THRESHOLDS = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]   # eV/Å

# ── Compute ───────────────────────────────────────────────────────────────────
F_DFT_fractions = build_F_dft_frac_table(all_results_filtered, FDFT_FRACS_THRESHOLDS)
display(F_DFT_fractions)

# ── Plot ──────────────────────────────────────────────────────────────────────
col_labels_fdft = [rf"$|F_{{DFT}}| > {thr:g}$ eV/Å" for thr in FDFT_FRACS_THRESHOLDS]

fig, ax = single_heatmap(
    data       = F_DFT_fractions.values,
    row_labels = F_DFT_fractions.index.tolist(),
    col_labels = col_labels_fdft,
    title      = (
        r"% atoms with $|F_{\mathrm{DFT}}|$ > threshold" "\n"
        "(all models see the same DFT forces)"
    ),
    cmap       = "plasma",
    annotate   = True,
    fmt        = "{:.1f}",
    textsize   = 8,
    addsize    = 0,
    cbar       = True,
    cbar_label = "[%]",
    figsize    = (4.0, 5.5),
    gap        = 0.025,
    right      = 0.80,
    bottom     = 0.02,
    top        = 1.0,
    savepath   = "./jsonfiles_for_summary_table/F_dft_frac_heatmap.svg",
)

---
## Section 17 — F_DFT-Conditioned MAE/RMSE of ΔF

For atoms with |F_DFT| **above** each threshold: MAE and RMSE of |ΔF|.

- Lower-left triangle  ← MAE
- Upper-right triangle ← RMSE
- Fraction header row  ← % of all atoms with |F_DFT| > threshold (avg across models)

Horizontal colorbars are placed below the table.

In [ ]:
from matpes_frac_analysis import (
    build_F_dft_conditioned_mae_rmse,
    merge_mae_rmse_as_string,
    triangular_heatmap_with_fraction_row_word_style,
)

# ── Config ────────────────────────────────────────────────────────────────────
FDFT_COND_THRESHOLDS = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 0.7, 1.0, 2.0]   # eV/Å

# ── Compute MAE / RMSE tables ─────────────────────────────────────────────────
F_DFT_df_cond_mae, F_DFT_df_cond_RMSE = build_F_dft_conditioned_mae_rmse(
    all_results_filtered, FDFT_COND_THRESHOLDS
)

print("MAE of |ΔF| (eV/Å)  conditioned on |F_DFT| > threshold")
display(F_DFT_df_cond_mae)
print("\nRMSE of |ΔF| (eV/Å)  conditioned on |F_DFT| > threshold")
display(F_DFT_df_cond_RMSE)

# ── Merged string table ───────────────────────────────────────────────────────
df_fdft_merged = merge_mae_rmse_as_string(F_DFT_df_cond_mae, F_DFT_df_cond_RMSE)
print("\nMAE / RMSE (eV/Å)")
display(df_fdft_merged)

# ── Fraction header row: avg % atoms with |F_DFT| > threshold ─────────────────
frac_cond_fdft = build_F_dft_frac_table(all_results_filtered, FDFT_COND_THRESHOLDS)
frac_row_17    = frac_cond_fdft.mean(axis=0).map(lambda x: f"{x:.1f}%")
frac_row_17.index = FDFT_COND_THRESHOLDS

# ── Plot ──────────────────────────────────────────────────────────────────────
col_labels_17 = (                                                                                                                                                                                           
      ["all atoms"]                                                                                                                                                                                           
      + [rf"$|F_{{DFT}}| > {thr:g}$ eV/Å" for thr in FDFT_COND_THRESHOLDS[1:]]                                                                                                                                
  )    


fig, ax = triangular_heatmap_with_fraction_row_word_style(
    mae_df       = F_DFT_df_cond_mae,
    rmse_df      = F_DFT_df_cond_RMSE,
    frac_row_str = frac_row_17,
    col_labels   = col_labels_17,
    title        = (
        r"(MAE / RMSE) of $|\Delta F|$ [eV/Å]" "\n"
        r"conditioned on $|F_{\mathrm{DFT}}| >$ threshold"
    ),
    cbar_height      = 0.025,   # ← thicker/thinner bars
    cbar_gap         = 0.2,   # ← gap: table bottom → MAE bar                                                                                                                                             
    cbar_between_gap = 0.1,   # ← gap: MAE bar → RMSE bar                                                                                                                                                 
    bottom           = 0.32,    # ← total figure bottom margin                                                                                                                                              
                                #   (increase if RMSE bar is clipped)   
    cmap_mae  = "Blues",
    cmap_rmse = "Reds",
    figsize   = (6.3, 8),
    fmt_mae   = "{:.2f}",
    fmt_rmse  = "{:.2f}",
    text_size = 12,
    savepath  = "./jsonfiles_for_summary_table/fdft_conditioned_mae_rmse.svg",
)

---
## Section 18 — Δθ Fraction Table

Fraction (%) of atoms with angular error **below** each threshold.  
Only atoms with |F_DFT| > FDFT_MIN are included.  
Displayed as an imshow-based heatmap.

In [ ]:
from matpes_frac_analysis import build_theta_frac_table, heatmap_fraction_delta_theta

# ── Config ────────────────────────────────────────────────────────────────────
THETA_THRESHOLDS = [1, 5, 10, 20, 30, 60, 90, 120, 178, 179]   # degrees

# ── Compute ───────────────────────────────────────────────────────────────────
DF_fractions = build_theta_frac_table(
    all_results_filtered, THETA_THRESHOLDS, fdf_min=FDFT_MIN
)
display(DF_fractions)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = heatmap_fraction_delta_theta(
    df_frac  = DF_fractions,
    title    = (
        r"% atoms with $|\Delta\theta| <$ threshold" "\n"
        rf"($|F_{{\mathrm{{DFT}}}}| > {FDFT_MIN}$ eV/Å only)"
    ),
    cbar_width = 0.015,   # ← thicker/thinner colorbar                                                                                                                                                      
    cbar_pad   = 0.01,    # ← gap between table and colorbar    
    cmap     = "viridis",
    fmt      = "{:.1f}",
    textsize = 8,
    figsize  = (5.5, 5.5),
    savepath = "./jsonfiles_for_summary_table/theta_frac_heatmap.svg",
)

---
## Section 19 — Δθ Conditioned MAE/RMSE

For atoms with |Δθ| **below** each threshold: MAE and RMSE of |Δθ|.

- Lower-left triangle  ← MAE (degrees)
- Upper-right triangle ← RMSE (degrees)
- Fraction header row  ← % of atoms passing the angular threshold

Uses `triangular_heatmap_with_fraction_row` from `heatmap_table.py` (vertical colorbars).

In [ ]:
import pandas as pd
from matpes_frac_analysis import build_theta_conditioned_mae_rmse, merge_mae_rmse_as_string
from heatmap_table import triangular_heatmap_with_fraction_row

# ── Config ────────────────────────────────────────────────────────────────────
THETA_COND_THRESHOLDS = [1, 5, 10, 20, 30, 60, 90, 120, 178, 179]   # degrees

# ── Compute MAE / RMSE tables ─────────────────────────────────────────────────
DF_COND_MAE, DF_df_cond_RMSE = build_theta_conditioned_mae_rmse(
    all_results_filtered, THETA_COND_THRESHOLDS, fdf_min=FDFT_MIN
)

print("MAE of |Δθ| (deg)  conditioned on |Δθ| < threshold")
display(DF_COND_MAE)
print("\nRMSE of |Δθ| (deg)  conditioned on |Δθ| < threshold")
display(DF_df_cond_RMSE)

# ── Merged string table ───────────────────────────────────────────────────────
df_theta_merged = merge_mae_rmse_as_string(DF_COND_MAE, DF_df_cond_RMSE)
print("\nMAE / RMSE (deg)")
display(df_theta_merged)

# ── Fraction header row: avg % atoms with |Δθ| < threshold ────────────────────
frac_row_19 = DF_fractions.mean(axis=0).map(lambda x: f"{x:.1f}%")
frac_row_19.index = THETA_COND_THRESHOLDS

# ── Plot ──────────────────────────────────────────────────────────────────────
col_labels_19 = [rf"$|\Delta\theta| < {thr}°$" for thr in THETA_COND_THRESHOLDS]

triangular_heatmap_with_fraction_row(
    mae_df       = DF_COND_MAE,
    rmse_df      = DF_df_cond_RMSE,
    frac_row_str = frac_row_19,
    col_labels   = col_labels_19,
    
    title        = (
        r"(MAE / RMSE) of $|\Delta\theta|$ [deg]" "\n"
        r"conditioned on $|\Delta\theta| <$ threshold"
    ),
    colorbar_width_ratio = 0.15,   # ← thicker/thinner colorbars                                                                                                                                            
    colorbar_nudge       = 0.05,   # ← gap between table and colorbars (larger = closer)
    wspace               = 0.3,   # ← gap between the two colorbars (larger = more space) 
    cmap_mae  = "Blues",
    cmap_rmse = "Reds",
    figsize   = (8.5, 8),
    fmt_mae   = "{:.2f}",
    fmt_rmse  = "{:.2f}",
    text_size = 11,
)

---
## Section 20 — Error Histograms

- **|ΔF| histogram**: log-x, log-y — shows the tail of large force errors
- **|Δθ| histogram**: linear-x, log-y — shows the angular error distribution (0°–180°)

In [ ]:
from matpes_frac_analysis import plot_error_histograms

# All atoms (|F_DFT| > FDFT_MIN), line style
plot_error_histograms(
    all_results_filtered,
    fdf_min        = FDFT_MIN,
    fdft_max       = None,     # no upper limit — all atoms
    bins_dF        = 100,
    bins_theta     = 90,
    bins_fdft      = 100,
    drawstyle      = "step",   # "line" or "step"
    figsize        = (5, 4),
    textsize       = 10,
    savepath_dF    = "./jsonfiles_for_summary_table/hist_abs_dF.svg",
    savepath_theta = "./jsonfiles_for_summary_table/hist_dtheta.svg",
    savepath_fdft  = "./jsonfiles_for_summary_table/hist_fdft.svg",
)


In [ ]:

# Section 20b — near-equilibrium only:
FDFT_HIST_MAX = 1.0   # eV/Å — upper |F_DFT| cutoff

plot_error_histograms(
    all_results_filtered,
    fdf_min        = FDFT_MIN,
    fdft_max       = FDFT_HIST_MAX,   # near-equilibrium atoms only
    bins_dF        = 100,
    bins_theta     = 90,
    bins_fdft      = 100,
    drawstyle      = "step",          # "line" or "step"
    figsize        = (5, 4),
    textsize       = 10,
    savepath_dF    = f"./jsonfiles_for_summary_table/hist_abs_dF_fdft_lt{FDFT_HIST_MAX}.svg",
    savepath_theta = f"./jsonfiles_for_summary_table/hist_dtheta_fdft_lt{FDFT_HIST_MAX}.svg",
    savepath_fdft  = f"./jsonfiles_for_summary_table/hist_fdft_fdft_lt{FDFT_HIST_MAX}.svg",
)

---
## Section 21 — Theta Regime Panels

Atoms split by |F_DFT| into two regimes:
- **Panel A** — near-equilibrium: |F_DFT| < 1 eV/Å
- **Panel B** — far-from-equilibrium: |F_DFT| ≥ 1 eV/Å

For each regime, fraction (%) of atoms with |Δθ| < threshold is shown.

In [ ]:
from matpes_frac_analysis import build_theta_regime_panels
from heatmap_table import plot_fraction_panel

# ── Config ────────────────────────────────────────────────────────────────────
THETA_REGIME_THRESH = [1, 5, 10, 20, 30, 60, 90, 120]   # degrees
THETA_THRESHOLD     = 1.0   # eV/Å boundary between panels

fig_w_th     = 6.5
fig_h_th     = 5.98
text_size_th = 13
cbar_gap_th  = 0.115
cbar_lbl_th  = 15

# ── Build DataFrames ─────────────────────────────────────────────────────────
df_panel_A_theta, df_panel_B_theta = build_theta_regime_panels(
    all_results_filtered,
    theta_thresh = THETA_REGIME_THRESH,
    threshold    = THETA_THRESHOLD,
    fdf_min      = FDFT_MIN,
)

print(f"Panel A — |F_DFT| < {THETA_THRESHOLD} eV/Å")
display(df_panel_A_theta)
print(f"Panel B — |F_DFT| ≥ {THETA_THRESHOLD} eV/Å")
display(df_panel_B_theta)

# Column subsets for plot_fraction_panel
theta_cols = (
    ["N atoms", "Frac of all atoms [%]"]
    + [f"|Δθ| < {thr} [%]" for thr in THETA_REGIME_THRESH]
)

frac_avg_A_theta = df_panel_A_theta["Frac of all atoms [%]"].mean()

# ── Panel A ───────────────────────────────────────────────────────────────────
plot_fraction_panel(
    df_panel_A_theta[theta_cols],
    fmt        = "{:.2f}",
    panel_title= (
        f"Near-equilibrium (|F_DFT| < {THETA_THRESHOLD} eV/Å) — Δθ fractions\n"
        f"({frac_avg_A_theta:.1f}% of all atoms)"
    ),
    fig_w=fig_w_th, fig_h=fig_h_th,
    text_size=text_size_th,
    cbar_gap=cbar_gap_th, cbar_label_pad=cbar_lbl_th,
    show_regime_row=False,
)

# ── Panel B ───────────────────────────────────────────────────────────────────
plot_fraction_panel(
    df_panel_B_theta[theta_cols],
    fmt        = "{:.2f}",
    panel_title= (
        f"Far-from-equilibrium (|F_DFT| ≥ {THETA_THRESHOLD} eV/Å) — Δθ fractions\n"
        f"({100 - frac_avg_A_theta:.1f}% of all atoms)"
    ),
    mae_cmap=plt.cm.Purples, rmse_cmap=plt.cm.Oranges,
    fig_w=fig_w_th, fig_h=fig_h_th,
    text_size=text_size_th,
    save_path="theta_regime_far_from_eq.svg",
    cbar_gap=cbar_gap_th, cbar_label_pad=cbar_lbl_th,
    show_regime_row=False,
)

## Querying bad atoms / structures by threshold

`get_bad_atom_indices(all_results, model, ...)` filters the **flat per-atom arrays** stored in `all_results` and returns:

- `atom_indices` — positions in the flat per-atom array
- `structure_indices` — `original_index` values of structures containing **at least one** matching atom

### Available filters (combine any/all freely)

| Argument | Meaning |
|---|---|
| `dF_gt=X` | |ΔF| > X eV/Å |
| `dF_lt=X` | |ΔF| < X eV/Å |
| `dtheta_gt=X` | |Δθ| > X degrees |
| `dtheta_lt=X` | |Δθ| < X degrees |
| `fdft_gt=X` | |F_DFT| > X eV/Å |
| `fdft_lt=X` | |F_DFT| < X eV/Å |

All conditions are combined with **AND** — only atoms satisfying every specified condition are returned.

In [ ]:
from matpes_frac_analysis import get_bad_atom_indices                                                                           
                                                                                                                                
MODEL = "mace"   # change to any model in all_results_filtered                                                                                                                                              
                                                                                                                                                                                                            
# ── Totals ────────────────────────────────────────────────────────────────────                                                                                                                            
d              = all_results_filtered[MODEL]                                                                                                                                                                
total_atoms    = len(d.get("all_deltaF", d.get("deltaF", [])))                                                                                                                                              
total_structs  = len(d.get("original_indices", []))                                                                                                                                                         
print(f"{'[TOTAL]':<35} {total_atoms:>7,} atoms  |  {total_structs:>5,} structures")                                                                                                                        
print("-" * 70)                                                                                                                                                                                             
                                                                                                                                                                                                            
def _report(label, atom_idx, struct_idx):                                                                                                                                                                   
    pa = 100 * len(atom_idx)   / total_atoms   if total_atoms   else 0                                                                                                                                      
    ps = 100 * len(struct_idx) / total_structs if total_structs else 0                                                                                                                                      
    print(f"{label:<35} {len(atom_idx):>7,} atoms ({pa:5.1f}%)  |  {len(struct_idx):>5,} structures ({ps:5.1f}%)")                                                                                          
                                                                                                                                                                                                            
# ── Example 1: |ΔF| > 0.1 eV/Å ──────────────────────────────────────────────                                                                                                                              
_report("[|ΔF| > 0.1]",                                                                                                                                                                                     
        *get_bad_atom_indices(all_results_filtered, MODEL, dF_gt=0.1))                                                                                                                                      

# ── Example 2: |ΔF| < 0.05 eV/Å (well-predicted atoms) ─────────────────────                                                                                                                               
_report("[|ΔF| < 0.05]",                                  
        *get_bad_atom_indices(all_results_filtered, MODEL, dF_lt=0.05))                                                                                                                                     
                                                                                                                                                                                                            
# ── Example 3: |Δθ| > 20° ────────────────────────────────────────────────────                                                                                                                             
_report("[|Δθ| > 20°]",                                                                                                                                                                                     
        *get_bad_atom_indices(all_results_filtered, MODEL, dtheta_gt=20))                                                                                                                                   
                                                        
# ── Example 4: |F_DFT| > 1.0 eV/Å (high-force regime) ──────────────────────                                                                                                                               
_report("[|F_DFT| > 1.0]",
        *get_bad_atom_indices(all_results_filtered, MODEL, fdft_gt=1.0))                                                                                                                                    
                                                                                                                                                                                                            
# ── Example 5: near-equilibrium AND large |ΔF| ───────────────────────────────                                                                                                                             
_report("[|F_DFT| < 1.0 AND |ΔF| > 0.1]",                                                                                                                                                                   
        *get_bad_atom_indices(all_results_filtered, MODEL, fdft_lt=1.0, dF_gt=0.1))                                                                                                                         
                                                                                                                                                                                                            
# ── Example 6: large angle AND large |ΔF| ────────────────────────────────────                                                                                                                             
_report("[|ΔF| > 0.1 AND |Δθ| > 20°]",                                                                                                                                                                      
        *get_bad_atom_indices(all_results_filtered, MODEL, dF_gt=0.1, dtheta_gt=20))                                                                                                                        
                                                                                                                                                                                                            
print("-" * 70)
atom_idx, struct_idx = get_bad_atom_indices(all_results_filtered, MODEL, dF_gt=0.1)                                                                                                                         
print("Original structure indices (first 10):", struct_idx[:10])  